# Joint LLM + ORBIT knowledge-graph reasoning

This notebook shows how a language model and the ORBIT knowledge graph are used **together**:

1. The model (or a fixed planner) selects structured graph tools.
2. The graph returns identifier-linked rows.
3. The model synthesizes a short answer that keeps sample / gene / reagent IDs.

The default path is **offline**: it replays the manuscript Prader-Willi case (`KM-14955`) from fixtures that match Data S5. Set `NEO4J_*` and an API key only if you want a live graph + live model.

In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if (ROOT / "orbit_kg").exists():
    MODULE = ROOT
elif (ROOT.parent / "orbit_kg").exists():
    MODULE = ROOT.parent
else:
    MODULE = ROOT.parents[0]
sys.path.insert(0, str(MODULE))

from orbit_kg import OrbitKGClient

FIXTURE = MODULE / "examples" / "fixtures" / "pws_km14955.json"
USE_LIVE_NEO4J = bool(os.getenv("NEO4J_URI") and os.getenv("NEO4J_PASSWORD"))
USE_LIVE_LLM = bool(os.getenv("OPENAI_API_KEY") or os.getenv("ANTHROPIC_API_KEY"))

client = OrbitKGClient(fixture_path=FIXTURE) if not USE_LIVE_NEO4J else OrbitKGClient()
print("graph mode:", client.mode)
print("llm mode:", "live" if USE_LIVE_LLM else "template (offline)")

## 1. Tool registry

Each tool is a named Cypher (or fixture) query. The LLM never invents edge types; it only chooses among these tools.

In [ ]:
TOOLS = {
    "sample_neighbourhood": {
        "description": "One-hop relationship counts for a Sample (sample_id).",
        "parameters": {"sample_id": "KM-14955"},
    },
    "phenotypes": {
        "description": "Curated phenotype records linked to a Sample.",
        "parameters": {"sample_id": "KM-14955"},
    },
    "dlk1_markers": {
        "description": "Count GroupInfo / ClusterMarkers edges for DLK1 in a Sample.",
        "parameters": {"sample_id": "KM-14955", "symbol": "DLK1"},
    },
    "radial_glia_markers": {
        "description": "Distinct radial-glial cluster marker symbols in GSE164102.",
        "parameters": {"gse_id": "GSE164102", "cluster": "Radial glial cells"},
    },
    "dlk1_annotation": {
        "description": "GeneAnnotation for DLK1 (STRING partner count, CRISPick designs).",
        "parameters": {"sample_id": "KM-14955", "symbol": "DLK1"},
    },
}

def call_tool(name: str):
    rows = client.run_named(name)
    return {"tool": name, "n_rows": len(rows), "rows": rows}

print("registered tools:", list(TOOLS))

## 2. Planner: LLM or deterministic joint plan

Offline mode uses the same tool order as the manuscript evidence chain (phenotype → cluster marker → annotation). Live mode asks the model to emit a JSON list of tool names.

In [ ]:
USER_QUESTION = (
    "For the Prader-Willi arcuate organoid sample KM-14955, identify a source-linked "
    "candidate gene connected to the retained-progenitor phenotype, then report "
    "interaction partners and CRISPRi/a/ko reagent counts."
)

DEFAULT_PLAN = [
    "sample_neighbourhood",
    "phenotypes",
    "radial_glia_markers",
    "dlk1_markers",
    "dlk1_annotation",
]

def plan_with_llm(question: str) -> list[str]:
    tool_doc = json.dumps(
        {k: v["description"] for k, v in TOOLS.items()}, indent=2
    )
    prompt = (
        "You jointly use an ORBIT knowledge-graph tool API. "
        "Return ONLY a JSON list of tool names, in execution order, chosen from:\n"
        f"{tool_doc}\n\nQuestion: {question}\n"
    )
    if os.getenv("OPENAI_API_KEY"):
        from openai import OpenAI

        model = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
        resp = OpenAI().chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": "Emit JSON only."},
                {"role": "user", "content": prompt},
            ],
            temperature=0,
        )
        text = resp.choices[0].message.content.strip()
        plan = json.loads(text)
        return [t for t in plan if t in TOOLS]
    raise RuntimeError("No OPENAI_API_KEY; use DEFAULT_PLAN")

plan = plan_with_llm(USER_QUESTION) if USE_LIVE_LLM else DEFAULT_PLAN
print("plan:", plan)

## 3. Execute tools against the graph

In [ ]:
evidence = [call_tool(name) for name in plan]
for step in evidence:
    print(f"\n### {step['tool']} ({step['n_rows']} rows)")
    print(json.dumps(step["rows"][:5], indent=2)[:800])

## 4. Synthesize an identifier-linked answer

The synthesizer may be a live LLM. Offline, we use a deterministic template that only cites fixture fields — the same provenance discipline as the manuscript KG trace.

In [ ]:
IMPRINTED = {"DLK1", "MEST"}

def synthesize_offline(blocks: list[dict]) -> str:
    by = {b["tool"]: b["rows"] for b in blocks}
    pheno = next(
        (r for r in by.get("phenotypes", []) if "NESTIN" in r.get("phenotype", "")),
        by.get("phenotypes", [{}])[0],
    )
    markers = [r["symbol"] for r in by.get("radial_glia_markers", [])]
    hits = [g for g in markers if g in IMPRINTED]
    ann = by.get("dlk1_annotation", [{}])[0]
    partners = ann.get("top_string_partners", [])
    partner_txt = ", ".join(f"{p['symbol']} ({p['score']})" for p in partners[:3])
    return (
        f"Sample KM-14955 links phenotype {pheno.get('node')} "
        f"(retained NESTIN+ progenitors) to the radial-glial ClusterMarkers set "
        f"in GSE164102 ({len(markers)} genes). Imprinted genes in that set: {', '.join(hits)}. "
        f"DLK1 (Entrez {ann.get('entrez')}) carries {ann.get('string_partners')} STRING partners "
        f"and {ann.get('crispick_designs')} CRISPick designs; top partners include {partner_txt}. "
        "This chain is identifier-linked; it does not claim that DLK1 causes the phenotype."
    )

def synthesize_with_llm(question: str, blocks: list[dict]) -> str:
    from openai import OpenAI

    model = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
    payload = json.dumps(blocks, indent=2)[:12000]
    prompt = (
        "Answer using ONLY the tool results. Keep sample_id, gene symbols, Entrez IDs, "
        "GSE accessions and reagent counts. Do not invent statistics.\n\n"
        f"Question: {question}\n\nTool results:\n{payload}\n"
    )
    resp = OpenAI().chat.completions.create(
        model=model,
        messages=[
            {
                "role": "system",
                "content": "You are a scientific assistant grounded in ORBIT KG tool output.",
            },
            {"role": "user", "content": prompt},
        ],
        temperature=0,
    )
    return resp.choices[0].message.content.strip()

answer = synthesize_with_llm(USER_QUESTION, evidence) if USE_LIVE_LLM else synthesize_offline(evidence)
print(answer)

## 5. What this demonstrates

| Layer | Role |
|-------|------|
| Language model | Plans tool calls and writes the narrative |
| Knowledge graph | Supplies deterministic, identifier-linked evidence |
| Joint loop | Model never replaces graph lookup for counts or IDs |

For live Cypher against Neo4j, see `examples/cypher/pws_km14955.cypher`. For construction, see `docs/BUILD.md`.

In [ ]:
client.close()